# Sanity check notebook 

This notebook is simply a sanaity check:

- generate a small synthetic DNA dataset with a planted motif
- test that hamming distance function works
- run each implemented algorithm once on the same dataset
- perform a small camprison across multiple runs 

This notebook is not used for the main synthetic evaluation or real-data evaluation.

In [ ]:
import sys
import os

project_root = os.path.abspath("..")
sys.path.append(project_root)

from src.data_utils import hamming, generate_dataset
from src.scoring import fitness_positions
from src.random_search import random_search
from src.rmhc import rmhc
from src.ga import genetic_algorithm
from src.aco import aco_motif_search
from src.simulated_annealing import simulated_annealing

### Generate small synthetic dataset
It would be a short sequence of DNA with a known motif planted

In [ ]:
#define motif and parameters
motif = "ACGTAC"
num_sequences = 5
sequence_length = 30
max_mismatches = 1
k = len(motif)
# generate the datsset with planted motifs
sequences, true_positions, planted_copies = generate_dataset(
    motif,
    num_sequences,
    sequence_length,
    max_mismatches
)

print("Original motif:", motif)
print()

for i in range(num_sequences):
    print("Sequence", i, ":", sequences[i])
    print("True position:", true_positions[i])
    print("Planted copy:", planted_copies[i], "Hamming:", hamming(motif, planted_copies[i]))
    print()

## Test fitness function works 
this is down by compaing random postions to true postions using hamming distance

In [ ]:
# generates random start positions of where the motif might be in each sequence & put it in a list

guess_positions = [0 for _ in range(num_sequences)]
# evaluate guessed postions 
score, consensus, kmers = fitness_positions(sequences, guess_positions, k)
#work out the hamming distance betweeen guessed motif and the actual motif
print("Guessed positions:", guess_positions)
print("Extracted k-mers:", kmers)
print("Consensus:", consensus)
print("Score:", score)
print()

true_score, true_consensus, true_kmers = fitness_positions(sequences, true_positions, k)

print("TRUE positions:", true_positions)
print("Extracted TRUE k-mers:", true_kmers)
print("Consensus from TRUE positions:", true_consensus)
print("Score for TRUE positions:", true_score)

## Run each algorithm once

In [ ]:
#RS algorithm 
best_positions, best_score, best_consensus = random_search(sequences, k, iterations=9000)
rs_distance = hamming(best_consensus, motif)
print("Random Search best consensus:", best_consensus)
print("Random Search Hamming distance:", rs_distance)
print("Random Search best positions:", best_positions)
print()


#rmhc
best_positions, best_score, best_consensus = rmhc(sequences, k, iterations=9000)
rmhc_distance = hamming(best_consensus, motif)
print("RMHC best consensus:", best_consensus)
print("RMHC Hamming distance:", rmhc_distance)
print("RMHC best positions:", best_positions)
print()

#GA
best_positions, best_score, best_consensus = genetic_algorithm(
    sequences,
    k,
    population_size=30,
    generations=200
)
ga_distance = hamming(best_consensus, motif)
print("GA best consensus:", best_consensus)
print("GA Hamming distance:", ga_distance)
print("GA best positions:", best_positions)
print()


#aco
best_positions, best_score, best_consensus = aco_motif_search(
    sequences,
    k,
    num_ants=20,
    iterations=100
)
aco_distance = hamming(best_consensus, motif)
print("ACO best consensus:", best_consensus)
print("ACO Hamming distance:", aco_distance)
print("ACO best positions:", best_positions)
print()

#sa
best_positions, best_score, best_consensus = simulated_annealing(
    sequences,
    k,
    iterations=5000,
    initial_temperature=10.0,
    cooling_rate=0.995
)
sa_distance = hamming(best_consensus, motif)
print("Simulated Annealing best consensus:", best_consensus)
print("Simulated Annealing Hamming distance:", sa_distance)
print("Simulated Annealing best positions:", best_positions)
print()

## repeated run

In [ ]:
runs = 10

#store the hamming distances for each algorithm
random_distances = []
rmhc_distances = []
ga_distances = []
aco_distances = []
sa_distances = []

for i in range(runs):
    #generate new dataset for every run
    sequences, true_positions, planted_copies = generate_dataset(
        motif,
        num_sequences,
        sequence_length,
        max_mismatches
    )
    
    _, _, rs_consensus = random_search(sequences, k, iterations=2000)
    random_distances.append(hamming(rs_consensus, motif))

    _, _, rmhc_consensus = rmhc(sequences, k, iterations=2000)
    rmhc_distances.append(hamming(rmhc_consensus, motif))

    _, _, ga_consensus = genetic_algorithm(
        sequences,
        k,
        population_size=30,
        generations=200
    )
    ga_distances.append(hamming(ga_consensus, motif))

    _, _, aco_consensus = aco_motif_search(
        sequences,
        k,
        num_ants=20,
        iterations=100
    )
    aco_distances.append(hamming(aco_consensus, motif))

    _, _, sa_consensus = simulated_annealing(
        sequences,
        k,
        iterations=2000,
        initial_temperature=10.0,
        cooling_rate=0.995
    )
    sa_distances.append(hamming(sa_consensus, motif))
#print the avraeg hamming distance 
print("Average Random Search Hamming distance:", sum(random_distances) / runs)
print("Average RMHC Hamming distance:", sum(rmhc_distances) / runs)
print("Average GA Hamming distance:", sum(ga_distances) / runs)
print("Average ACO Hamming distance:", sum(aco_distances) / runs)
print("Average SA Hamming distance:", sum(sa_distances) / runs)